## 1. Setup

In [1]:
## Cell 1
import time
import io
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import requests
import yfinance as yf
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
print(f"yfinance version: {yf.__version__}")


yfinance version: 1.7.0


## 2. Configuration

In [2]:
# Cell 2
START_DATE = "1990-01-01"
END_DATE = "2015-10-31"

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TICKER_LIMIT = None

REQUEST_SLEEP_SECONDS = 0.6
MAX_RETRIES = 2
RETRY_BACKOFF_SECONDS = 5

HISTORICAL_CONSTITUENTS_URL = (
    "https://raw.githubusercontent.com/fja05680/sp500/master/"
    "S%26P%20500%20Historical%20Components%20%26%20Changes%20(Updated).csv"
)

print(f"Date range:  {START_DATE} to {END_DATE}")
print(f"Ticker limit: {TICKER_LIMIT if TICKER_LIMIT else 'full expanded universe'}")


Date range:  1990-01-01 to 2015-10-31
Ticker limit: full expanded universe


## 3. Current constituents and company reference table

(Same as the original notebook: today's list, with GICS sector/industry,
scraped from Wikipedia.)


In [3]:
## Cell 3
def get_sp500_reference_table() -> pd.DataFrame:
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                              "AppleWebKit/537.36 (KHTML, like Gecko) "
                              "Chrome/120.0.0.0 Safari/537.36")}
    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()
    tables = pd.read_html(io.StringIO(resp.text))
    table = tables[0].copy()
    table["Symbol"] = table["Symbol"].str.replace(".", "-", regex=False)
    table = table.rename(columns={
        "Symbol": "ticker", "Security": "company_name",
        "GICS Sector": "gics_sector", "GICS Sub-Industry": "gics_sub_industry",
        "Date added": "index_date_added", "Founded": "founded",
    })
    keep = [c for c in ["ticker", "company_name", "gics_sector", "gics_sub_industry",
                        "index_date_added", "founded"] if c in table.columns]
    return table[keep]

current_reference = get_sp500_reference_table()
current_tickers = set(current_reference["ticker"])
print(f"Current S&P 500 constituents (Wikipedia): {len(current_tickers)}")


Current S&P 500 constituents (Wikipedia): 503


## 4. Historical constituents (NEW) — expanding the universe to include delisted names

Downloads the community-maintained historical membership file and computes
the **union of every ticker that was an S&P 500 constituent on any date**
within the paper's period. This union is the expanded download universe.


In [4]:
## Cell 4
def get_historical_universe(start: str, end: str) -> tuple[set, pd.DataFrame]:
    resp = requests.get(HISTORICAL_CONSTITUENTS_URL, timeout=60)
    resp.raise_for_status()

    raw = pd.read_csv(io.StringIO(resp.text))
    raw.columns = [c.lower() for c in raw.columns]
    raw["date"] = pd.to_datetime(raw["date"])

    period = raw[(raw["date"] >= start) & (raw["date"] <= end)].copy()
    period.to_csv(PROCESSED_DIR / "historical_constituents_raw.csv", index=False)

    universe = set()
    for tickers_str in period["tickers"]:
        universe.update(t.strip() for t in tickers_str.split(","))

    return universe, period


historical_universe, historical_period_df = get_historical_universe(START_DATE, END_DATE)

print(f"Historical file's actual coverage within the requested period: "
      f"{historical_period_df['date'].min().date()} to {historical_period_df['date'].max().date()}")
print(f"(Requested: {START_DATE} to {END_DATE} -- earlier dates are simply "
      f"not covered by this free source; see the limitation note in Section 0.)")
print(f"\nDistinct tickers that were ever an S&P 500 constituent in this window: "
      f"{len(historical_universe)}")

delisted_only = historical_universe - current_tickers
print(f"Of these, {len(delisted_only)} are NOT in today's constituent list "
      f"(i.e. historically delisted, renamed, or acquired).")


Historical file's actual coverage within the requested period: 1996-01-02 to 2015-10-28
(Requested: 1990-01-01 to 2015-10-31 -- earlier dates are simply not covered by this free source; see the limitation note in Section 0.)

Distinct tickers that were ever an S&P 500 constituent in this window: 978
Of these, 647 are NOT in today's constituent list (i.e. historically delisted, renamed, or acquired).


In [5]:
## Cell 5

full_universe = sorted(current_tickers | historical_universe)

company_info = current_reference.copy()
company_info["currently_listed"] = True

historical_only_df = pd.DataFrame({
    "ticker": sorted(delisted_only),
    "company_name": None, "gics_sector": None, "gics_sub_industry": None,
    "index_date_added": None, "founded": None, "currently_listed": False,
})
company_info = pd.concat([company_info, historical_only_df], ignore_index=True)
company_info.to_csv(PROCESSED_DIR / "company_info.csv", index=False)

print(f"Final download universe: {len(full_universe)} tickers "
      f"({len(current_tickers)} currently listed + {len(delisted_only)} historical-only)")

tickers = full_universe
if TICKER_LIMIT:
    tickers = tickers[:TICKER_LIMIT]
    print(f"TICKER_LIMIT set: using only the first {len(tickers)} for this run.")


Final download universe: 1150 tickers (503 currently listed + 647 historical-only)


## 6. Price history download function (unchanged from the original notebook)

In [6]:
# Cell 6
def download_ticker_history(ticker: str, start: str, end: str,
                             max_retries: int = MAX_RETRIES) -> pd.DataFrame | None:
    for attempt in range(max_retries + 1):
        try:
            df = yf.download(ticker, start=start, end=end, auto_adjust=True,
                             actions=True, progress=False, threads=False)
            if df is None or df.empty:
                return None
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = [c[0] for c in df.columns]
            df = df.reset_index()
            df.columns = [str(c).lower().replace(" ", "_") for c in df.columns]
            for col in ["dividends", "stock_splits"]:
                if col not in df.columns:
                    df[col] = 0.0
            return df[["date", "open", "high", "low", "close", "volume",
                       "dividends", "stock_splits"]]
        except Exception as e:
            if attempt < max_retries:
                time.sleep(RETRY_BACKOFF_SECONDS)
                continue
            print(f"  [FAILED] {ticker}: {e}")
            return None

print("Download function defined.")


Download function defined.


## 7. Run the download loop across the expanded universe

This will take noticeably longer than the original run (roughly twice as
many tickers), and a **higher failure rate is expected and correct** --
it is specifically measuring how many historically-delisted names can
still be recovered from a free source.


In [7]:
# Cell 7
download_log = []
for i, ticker in enumerate(tickers):
    print(f"[{i+1}/{len(tickers)}] {ticker}...", end=" ")
    df = download_ticker_history(ticker, START_DATE, END_DATE)
    if df is not None and len(df) > 0:
        df.to_csv(RAW_DIR / f"{ticker}.csv", index=False)
        print(f"OK ({len(df)} rows)")
        download_log.append({"ticker": ticker, "status": "ok", "rows": len(df),
                             "first_date": df["date"].min(), "last_date": df["date"].max()})
    else:
        print("FAILED")
        download_log.append({"ticker": ticker, "status": "failed", "rows": 0,
                             "first_date": None, "last_date": None})
    time.sleep(REQUEST_SLEEP_SECONDS)

download_log_df = pd.DataFrame(download_log)
download_log_df = download_log_df.merge(
    company_info[["ticker", "currently_listed"]], on="ticker", how="left")

print(f"\nOverall: {(download_log_df['status']=='ok').sum()}/{len(download_log_df)} succeeded.")


[1/1150] A... OK (4013 rows)
[2/1150] AABA... 

$AABA: possibly delisted; no timezone found

1 Failed download:
['AABA']: possibly delisted; no timezone found


FAILED
[3/1150] AAL... OK (2542 rows)
[4/1150] AAMRQ... 

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AAMRQ"}}}
$AAMRQ: possibly delisted; no timezone found

1 Failed download:
['AAMRQ']: possibly delisted; no timezone found


FAILED
[5/1150] AAP... OK (3505 rows)
[6/1150] AAPL... OK (6511 rows)
[7/1150] ABBV... OK (714 rows)
[8/1150] ABC... 

...

FAILED
[1144/1150] YNR... 

$YNR: possibly delisted; no timezone found

1 Failed download:
['YNR']: possibly delisted; no timezone found


FAILED
[1145/1150] YRCW... 

$YRCW: possibly delisted; no timezone found

1 Failed download:
['YRCW']: possibly delisted; no timezone found


FAILED
[1146/1150] YUM... OK (4561 rows)
[1147/1150] ZBH... OK (3590 rows)
[1148/1150] ZBRA... OK (6101 rows)
[1149/1150] ZION... OK (6511 rows)
[1150/1150] ZTS... OK (693 rows)

Overall: 609/1150 succeeded.


## 8. The key result: success rate by group

This is the number that actually answers "did expanding the universe help":
compare the success rate for currently-listed tickers (expected high,
matches the original notebook) against historical-only tickers (expected
much lower, since free sources rarely retain delisted-stock history).


In [8]:
# Cell 8
by_group = download_log_df.groupby("currently_listed")["status"].value_counts(normalize=False).unstack(fill_value=0)
by_group["success_rate"] = by_group.get("ok", 0) / by_group.sum(axis=1)
by_group.index = by_group.index.map({True: "Currently listed", False: "Historical-only (delisted/renamed/acquired)"})
print(by_group)

n_delisted_recovered = int(((download_log_df["currently_listed"] == False) &
                            (download_log_df["status"] == "ok")).sum())
n_delisted_attempted = int((download_log_df["currently_listed"] == False).sum())
print(f"\n{n_delisted_recovered} of {n_delisted_attempted} historically-delisted tickers "
      f"had SOME recoverable price history from Yahoo Finance.")
if n_delisted_recovered > 0:
    recovered_examples = download_log_df[(download_log_df["currently_listed"] == False) &
                                         (download_log_df["status"] == "ok")]["ticker"].tolist()
    print(f"Examples: {recovered_examples[:20]}")


status                                       failed   ok  success_rate
currently_listed                                                      
Historical-only (delisted/renamed/acquired)     502  145      0.224111
Currently listed                                 39  464      0.922465

145 of 647 historically-delisted tickers had SOME recoverable price history from Yahoo Finance.
Examples: ['AAL', 'AAP', 'ACV', 'AIT', 'AIV', 'AMG', 'AMH', 'AN', 'ANF', 'AR', 'ASC', 'ASH', 'ASND', 'ATI', 'BBT', 'BC', 'BCO', 'BUD', 'BWA', 'CAG']


## 9. Benchmark index — S&P 500 (`^GSPC`) (unchanged)

In [9]:
# Cell 9
spx = download_ticker_history("^GSPC", START_DATE, END_DATE)
if spx is not None:
    spx.to_csv(RAW_DIR / "_SPX_benchmark.csv", index=False)
    print(f"Benchmark saved: {len(spx)} rows")
else:
    print("WARNING: benchmark index download failed.")


Benchmark saved: 6511 rows


## 10. Data-quality diagnostics (now split by currently-listed vs. historical-only)

In [10]:
# Cell 10
def compute_coverage_diagnostics(ticker: str) -> dict:
    path = RAW_DIR / f"{ticker}.csv"
    if not path.exists():
        return {"ticker": ticker, "rows": 0, "first_date": None, "last_date": None,
                "coverage_ratio": 0.0, "max_gap_days": None}
    df = pd.read_csv(path, parse_dates=["date"]).sort_values("date")
    if len(df) == 0:
        return {"ticker": ticker, "rows": 0, "first_date": None, "last_date": None,
                "coverage_ratio": 0.0, "max_gap_days": None}
    expected_bdays = len(pd.bdate_range(START_DATE, END_DATE))
    gaps = df["date"].diff().dt.days.dropna()
    return {
        "ticker": ticker, "rows": len(df),
        "first_date": df["date"].min().date(), "last_date": df["date"].max().date(),
        "coverage_ratio": round(len(df) / expected_bdays, 4) if expected_bdays > 0 else np.nan,
        "max_gap_days": int(gaps.max()) if len(gaps) > 0 else None,
    }

diagnostics_df = pd.DataFrame([compute_coverage_diagnostics(t) for t in tickers])
diagnostics_df = diagnostics_df.merge(company_info[["ticker", "currently_listed"]], on="ticker", how="left")
diagnostics_df.to_csv(PROCESSED_DIR / "data_availability_report.csv", index=False)

print("Coverage summary by group:")
print(diagnostics_df.groupby("currently_listed")["coverage_ratio"].describe())


Coverage summary by group:
                  count      mean       std  min     25%     50%    75%    max
currently_listed                                                              
False             647.0  0.149670  0.318266  0.0  0.0000  0.0000  0.000  0.966
True              503.0  0.688398  0.344814  0.0  0.4208  0.8832  0.966  0.966


## 11. Consolidated processed panels (unchanged logic)

In [11]:
# Cell 11
price_series = {}
actions_records = []
for ticker in tickers:
    path = RAW_DIR / f"{ticker}.csv"
    if not path.exists():
        continue
    df = pd.read_csv(path, parse_dates=["date"])
    price_series[ticker] = df.set_index("date")["close"]
    events = df[(df["dividends"] > 0) | (df["stock_splits"] > 0)]
    for _, row in events.iterrows():
        actions_records.append({"ticker": ticker, "date": row["date"],
                                "dividend": row["dividends"], "split_ratio": row["stock_splits"]})

price_panel = pd.DataFrame(price_series).sort_index()
price_panel.to_csv(PROCESSED_DIR / "price_panel.csv")
pd.DataFrame(actions_records).to_csv(PROCESSED_DIR / "corporate_actions.csv", index=False)

print(f"Price panel: {price_panel.shape[0]} dates x {price_panel.shape[1]} tickers")


Price panel: 6511 dates x 609 tickers
